# Notebook 02: Classification Heads on Frozen Encoder

Goal of this notebook: Evaluate different classification head architectures on top of the Moirai foundation model. 

We evaluate four distinct architectures:
1. Mean Pooling
2. Single-Scale Attention
3. Single-Scale Multi-Head Attention (MHA)
4. Hierarchical Multi-Head Attention

In [1]:
import torch
import pandas as pd
from IPython.display import display

from uni2ts.model.moirai import MoiraiModule
from encoder import MoiraiEncoder
from heads import (
    MeanPoolingClassifier, 
    SingleScaleAttentionClassifier, 
    SingleScaleMultiHeadClassifier, 
    HierarchicalMultiHeadClassifier
)
from utils import get_lsst_dataloaders, get_z_loaders, grid_search_heads

# Global Configuration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_VARS = 6
SIZE = "small"
PATCH_SIZES_TO_TEST = [8, 16, 32, 64]
HEADS_TO_TEST = [4, 8, 16, 32, 64, 128, 384] 

# Hyperparameters
MOIRAI_BATCH_SIZE = 32
HEAD_BATCH_SIZE = 256
LR_GRID = [5*1e-4]
WD_GRID = [0.05, 0.01]
MODES = ["shared_context", "independent_context"]


df_patch_8_metrics = pd.DataFrame(index=[], columns= ['Accuracy', 'Macro Precision', 'Macro Recall', 'Macro F1',
       'Weighted Precision', 'Weighted Recall', 'Weighted F1'])


/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
raw_tr, raw_va, raw_te, num_classes = get_lsst_dataloaders(batch_size=MOIRAI_BATCH_SIZE, device=DEVICE)

## The Fast Evaluation Loop
Since Moirai is frozen, we loop over the patch sizes, instantiate Moirai, and **pre-extract the embeddings ($Z$)**. We then pass these $Z$ loaders to the different head architectures.

In [3]:
# We define a function to run the evaluation for a specific patch size
def evaluate_heads_for_patch(patch):
    
    # 1. Instantiate the Frozen Encoder
    encoder = MoiraiEncoder(
        module=MoiraiModule.from_pretrained(f"Salesforce/moirai-1.1-R-{SIZE}"),
        prediction_length=patch, context_length=36, patch_size=patch, 
        num_samples=100, target_dim=NUM_VARS, feat_dynamic_real_dim=0, past_feat_dynamic_real_dim=0,
    )
    
    # 2. Pre-Extract Embeddings (Z)
    tr_z, va_z, te_z = get_z_loaders(
        encoder, raw_tr, raw_va, raw_te, 
        head_batch_size=HEAD_BATCH_SIZE, device=DEVICE, remove_last_patch=False, num_vars=NUM_VARS
    )
    
    return tr_z, va_z, te_z

## 1. Mean Pooling Architecture


How it works: The simplest baseline. It takes the contextual patches generated by Moirai and averages them over the temporal dimension. The result is a single flattened vector containing the average representation of each variable, which is then passed to a Linear classifier.

In [4]:
results_mean_pooling = {}

for patch in PATCH_SIZES_TO_TEST:
    tr_z, va_z, te_z = evaluate_heads_for_patch(patch) # Extract once
    
    _, metrics = grid_search_heads(
        MeanPoolingClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes},
        tr_z, va_z, te_z, lr_grid=LR_GRID, wd_grid=WD_GRID, device=DEVICE
    )
    results_mean_pooling[patch] = metrics

In [5]:
df_results_mean_pooling = pd.DataFrame.from_dict(results_mean_pooling, orient='index')
df_results_mean_pooling.index.name = 'Patch Size'
print('results mean pooling')
display(df_results_mean_pooling[['Accuracy']])
df_patch_8_metrics.loc['Mean Pooling'] = df_results_mean_pooling.loc[8]
print('Patch  size = 8')
display(df_patch_8_metrics)

results mean pooling


,Accuracy
Patch Size,
8,0.615166
16,0.616788
32,0.607461
64,0.635442


Patch  size = 8


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Mean Pooling,0.615166,0.528663,0.378291,0.399711,0.571376,0.615166,0.571516


## 2. Single-Scale Attention Architecture


How it works: Instead of a naive average, this head uses a learned Query vector. This Query looks at all the patches (Keys/Values) and assigns an attention weight to each. The network learns which patches are the most important for the classification task.
* shared_context: One single Query acts across all variables simultaneously.
* independent_context: Each variable has its own dedicated Query to find its own important patches.

In [6]:
results_attn = {}

for patch in PATCH_SIZES_TO_TEST:
    tr_z, va_z, te_z = evaluate_heads_for_patch(patch)
    results_attn[patch] = {}

    for mode in MODES:
        _, metrics = grid_search_heads(
            SingleScaleAttentionClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "mode": mode},
            tr_z, va_z, te_z, lr_grid=LR_GRID, wd_grid=WD_GRID, device=DEVICE
        )
        results_attn[patch][mode] = metrics
        if patch == 8:
            df_patch_8_metrics.loc[f"Attention ({mode})"] = metrics

# Accuracy only for all patch sizes
df_attn_acc = pd.DataFrame(
    {patch: {mode: results_attn[patch][mode]['Accuracy'] for mode in MODES} for patch in PATCH_SIZES_TO_TEST}
)
df_attn_acc.index.name = 'Mode'
df_attn_acc.columns.name = 'Patch Size'
print('Results Single-Scale Attention - Accuracy')
display(df_attn_acc.astype(float).round(4))

print('Patch size = 8')
display(df_patch_8_metrics)

Results Single-Scale Attention - Accuracy


Patch Size,8,16,32,64
Mode,,,,
shared_context,0.6030,0.5835,0.5835,0.6298
independent_context,0.5994,0.6046,0.5937,0.6318


Patch size = 8


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Mean Pooling,0.615166,0.528663,0.378291,0.399711,0.571376,0.615166,0.571516
Attention (shared_context),0.603001,0.498070,0.365579,0.381272,0.550334,0.603001,0.552273
Attention (independent_context),0.599351,0.426421,0.358599,0.369874,0.545476,0.599351,0.553098


## 3. Multi-Head Attention (MHA)


How it works: A single attention mechanism might focus entirely on one feature (e.g., the highest peak). Multi-Head Attention solves this by projecting the data into multiple sub-spaces (e.g., 16 heads).
This allows the model to capture multiple distinct temporal concepts simultaneously (e.g., Head 1 looks at recent patches, Head 2 looks at periodic drops, ...).

### 3.1 MHA with 16 heads over all patch sizes

In [7]:
results_mha = {}

for patch in PATCH_SIZES_TO_TEST:
    tr_z, va_z, te_z = evaluate_heads_for_patch(patch)
    results_mha[patch] = {}

    for mode in MODES:
        _, metrics = grid_search_heads(
            SingleScaleMultiHeadClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": 16},
            tr_z, va_z, te_z, lr_grid=LR_GRID, wd_grid=WD_GRID, device=DEVICE
        )
        results_mha[patch][mode] = metrics

# Accuracy only for all patch sizes
df_mha_acc = pd.DataFrame(
    {patch: {mode: results_mha[patch][mode]['Accuracy'] for mode in MODES} for patch in PATCH_SIZES_TO_TEST}
)
df_mha_acc.index.name = 'Mode'
df_mha_acc.columns.name = 'Patch Size'
print('Results MHA (16 heads) - Accuracy')
display(df_mha_acc.astype(float).round(4))

Results MHA (16 heads) - Accuracy


Patch Size,8,16,32,64
Mode,,,,
shared_context,0.6107,0.6294,0.5969,0.6277
independent_context,0.6144,0.6212,0.5953,0.6249


### 3.2 MHA over patches of sizes 8 but with highter number of heads

In [8]:
results_mha8 = {}

for heads in HEADS_TO_TEST:
    tr_z, va_z, te_z = evaluate_heads_for_patch(8)
    results_mha8[heads] = {}

    for mode in MODES:
        _, metrics = grid_search_heads(
            SingleScaleMultiHeadClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": heads},
            tr_z, va_z, te_z, lr_grid=LR_GRID, wd_grid=WD_GRID, device=DEVICE
        )
        results_mha8[heads][mode] = metrics
        df_patch_8_metrics.loc[f"MHA-{heads} ({mode})"] = metrics

In [9]:
# Accuracy by number of heads (all runs are patch=8)
df_mha8_acc = pd.DataFrame(
    {heads: {mode: results_mha8[heads][mode]['Accuracy'] for mode in MODES} for heads in HEADS_TO_TEST}
)
df_mha8_acc.index.name = 'Mode'
df_mha8_acc.columns.name = 'Num Heads'
print('Results MHA (Patch = 8) - Accuracy by number of heads')
display(df_mha8_acc.astype(float).round(4))

print('Patch size = 8')
display(df_patch_8_metrics)

Results MHA (Patch = 8) - Accuracy by number of heads


Num Heads,4,8,16,32,64,128,384
Mode,,,,,,,
shared_context,0.6123,0.6046,0.6115,0.6103,0.6127,0.6229,0.6018
independent_context,0.6135,0.6119,0.6054,0.6119,0.6119,0.6144,0.6123


Patch size = 8


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Mean Pooling,0.615166,0.528663,0.378291,0.399711,0.571376,0.615166,0.571516
Attention (shared_context),0.603001,0.498070,0.365579,0.381272,0.550334,0.603001,0.552273
Attention (independent_context),0.599351,0.426421,0.358599,0.369874,0.545476,0.599351,0.553098
MHA-4 (shared_context),0.612328,0.419980,0.363378,0.369732,0.556705,0.612328,0.557157
MHA-4 (independent_context),0.613544,0.516471,0.387067,0.405486,0.566424,0.613544,0.571455
MHA-8 (shared_context),0.604623,0.436460,0.367592,0.377049,0.556295,0.604623,0.561372
MHA-8 (independent_context),0.611922,0.495860,0.379067,0.393483,0.559850,0.611922,0.566907
MHA-16 (shared_context),0.611517,0.443878,0.370081,0.380898,0.561108,0.611517,0.569818
MHA-16 (independent_context),0.605434,0.426609,0.370817,0.377788,0.552792,0.605434,0.567081
MHA-32 (shared_context),0.610300,0.502826,0.363124,0.374556,0.566421,0.610300,0.560484


## 4. Hierarchical Multi-Head Attention


How it works: Inspired by Hierarchical Attention Networks (HAN). It performs attention in two steps:
1. Temporal Attention: MHA is applied *within* each variable individually to summarize its temporal patches into a single variable-vector.
2. Variable Attention: A second MHA is applied *across* the summarized variables. This allows the network to learn inter-variable correlations.

In [10]:
results_hier = {}

for patch in PATCH_SIZES_TO_TEST:
    tr_z, va_z, te_z = evaluate_heads_for_patch(patch)
    results_hier[patch] = {}

    for mode in MODES:
        _, metrics = grid_search_heads(
            HierarchicalMultiHeadClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": 16},
            tr_z, va_z, te_z, lr_grid=LR_GRID, wd_grid=WD_GRID, device=DEVICE
        )
        results_hier[patch][mode] = metrics
        if patch == 8:
            df_patch_8_metrics.loc[f"Hierarchical MHA ({mode})"] = metrics

df_hier_acc = pd.DataFrame(
    {patch: {mode: results_hier[patch][mode]['Accuracy'] for mode in MODES} for patch in PATCH_SIZES_TO_TEST}
)
df_hier_acc.index.name = 'Mode'
df_hier_acc.columns.name = 'Patch Size'
print('Results Hierarchical MHA (16 heads) - Accuracy')
display(df_hier_acc.astype(float).round(4))

print('Patch size = 8')
display(df_patch_8_metrics)

Results Hierarchical MHA (16 heads) - Accuracy


Patch Size,8,16,32,64
Mode,,,,
shared_context,0.5799,0.5823,0.5722,0.5766
independent_context,0.6050,0.5921,0.5827,0.5896


Patch size = 8


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Mean Pooling,0.615166,0.528663,0.378291,0.399711,0.571376,0.615166,0.571516
Attention (shared_context),0.603001,0.498070,0.365579,0.381272,0.550334,0.603001,0.552273
Attention (independent_context),0.599351,0.426421,0.358599,0.369874,0.545476,0.599351,0.553098
MHA-4 (shared_context),0.612328,0.419980,0.363378,0.369732,0.556705,0.612328,0.557157
MHA-4 (independent_context),0.613544,0.516471,0.387067,0.405486,0.566424,0.613544,0.571455
MHA-8 (shared_context),0.604623,0.436460,0.367592,0.377049,0.556295,0.604623,0.561372
MHA-8 (independent_context),0.611922,0.495860,0.379067,0.393483,0.559850,0.611922,0.566907
MHA-16 (shared_context),0.611517,0.443878,0.370081,0.380898,0.561108,0.611517,0.569818
MHA-16 (independent_context),0.605434,0.426609,0.370817,0.377788,0.552792,0.605434,0.567081
MHA-32 (shared_context),0.610300,0.502826,0.363124,0.374556,0.566421,0.610300,0.560484


## Final Summary
Review of all architectures evaluated on the frozen Moirai encoder.

In [11]:
print("FINAL SUMMARY — Patch Size = 8 (All Metrics)")
display(df_patch_8_metrics.astype(float).round(4))

print("\nACCURACY ACROSS PATCH SIZES")

print("\nMean Pooling")
display(df_results_mean_pooling[['Accuracy']].astype(float).round(4))

print("\nSingle-Scale Attention")
display(df_attn_acc.astype(float).round(4))

print("\nMHA (16 heads)")
display(df_mha_acc.astype(float).round(4))

print("\nMHA (Patch = 8, varying heads)")
display(df_mha8_acc.astype(float).round(4))

print("\nHierarchical MHA (16 heads)")
display(df_hier_acc.astype(float).round(4))

FINAL SUMMARY — Patch Size = 8 (All Metrics)


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Mean Pooling,0.6152,0.5287,0.3783,0.3997,0.5714,0.6152,0.5715
Attention (shared_context),0.6030,0.4981,0.3656,0.3813,0.5503,0.6030,0.5523
Attention (independent_context),0.5994,0.4264,0.3586,0.3699,0.5455,0.5994,0.5531
MHA-4 (shared_context),0.6123,0.4200,0.3634,0.3697,0.5567,0.6123,0.5572
MHA-4 (independent_context),0.6135,0.5165,0.3871,0.4055,0.5664,0.6135,0.5715
MHA-8 (shared_context),0.6046,0.4365,0.3676,0.3770,0.5563,0.6046,0.5614
MHA-8 (independent_context),0.6119,0.4959,0.3791,0.3935,0.5598,0.6119,0.5669
MHA-16 (shared_context),0.6115,0.4439,0.3701,0.3809,0.5611,0.6115,0.5698
MHA-16 (independent_context),0.6054,0.4266,0.3708,0.3778,0.5528,0.6054,0.5671
MHA-32 (shared_context),0.6103,0.5028,0.3631,0.3746,0.5664,0.6103,0.5605



ACCURACY ACROSS PATCH SIZES

Mean Pooling


,Accuracy
Patch Size,
8,0.6152
16,0.6168
32,0.6075
64,0.6354



Single-Scale Attention


...

Patch Size,8,16,32,64
Mode,,,,
shared_context,0.6107,0.6294,0.5969,0.6277
independent_context,0.6144,0.6212,0.5953,0.6249



MHA (Patch = 8, varying heads)


Num Heads,4,8,16,32,64,128,384
Mode,,,,,,,
shared_context,0.6123,0.6046,0.6115,0.6103,0.6127,0.6229,0.6018
independent_context,0.6135,0.6119,0.6054,0.6119,0.6119,0.6144,0.6123



Hierarchical MHA (16 heads)


Patch Size,8,16,32,64
Mode,,,,
shared_context,0.5799,0.5823,0.5722,0.5766
independent_context,0.6050,0.5921,0.5827,0.5896
